Carga de librerías y definir ruta

In [2]:
import os
import re
import sys
import subprocess
import importlib
import warnings

def install_if_missing(package_name, import_name=None):
    """Verifica si un paquete está instalado; si no, lo instala automáticamente."""
    if import_name is None: 
        import_name = package_name
    try:
        importlib.import_module(import_name)
        print(f"✅ {import_name} ya está instalado.")
    except ImportError:
        print(f"📦 {import_name} no encontrado. Instalando {package_name}...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "--user", package_name])
        print(f"✅ {package_name} instalado correctamente.")

# Lista de TODAS las librerías externas que usa este script
dependencias = [
    ("pandas", "pandas"),
    ("geopandas", "geopandas"),
    ("transformers", "transformers"),
    ("tf-keras", "tf_keras"),   
    ("torch", "torch")         
]

print("🔄 Verificando e instalando dependencias (puede tardar si falta alguna)...")
for pkg, imp in dependencias:
    install_if_missing(pkg, imp)


import pandas as pd
import geopandas as gpd
from transformers import pipeline

warnings.filterwarnings('ignore')  

 
if os.path.exists('data'):
    PATH_DATA = 'data'
elif os.path.exists(os.path.join('..', 'data')):
    PATH_DATA = os.path.join('..', 'data')
else:
    PATH_DATA = 'data'

print(f"📂 Directorio de datos configurado en: {os.path.abspath(PATH_DATA)}")

🔄 Verificando e instalando dependencias (puede tardar si falta alguna)...
✅ pandas ya está instalado.
✅ geopandas ya está instalado.


c:\Users\delga\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


✅ transformers ya está instalado.

✅ tf_keras ya está instalado.
✅ torch ya está instalado.
📂 Directorio de datos configurado en: c:\Users\delga\OneDrive\Desktop\Intelencia Artificial\Trabajo-Inteligencia-Artificial\data


ETL espacial

In [3]:
ruta_shp_espana = os.path.join(PATH_DATA, 'SECC_CE_20230101.shp')
ruta_geojson_sevilla = os.path.join(PATH_DATA, 'mapa_sevilla.geojson')

if os.path.exists(ruta_shp_espana):
    mapa_espana = gpd.read_file(ruta_shp_espana)
    mapa_sevilla = mapa_espana[mapa_espana['CUSEC'].astype(str).str.startswith('41091')].copy()
    mapa_sevilla.to_file(ruta_geojson_sevilla, driver='GeoJSON')
    print(f" Mapa creado con {len(mapa_sevilla)} secciones.")
else:
    print(f"Error: No se encuentra {ruta_shp_espana}")

 Mapa creado con 521 secciones.


ETL del archivo listings.csv

In [4]:
  # Definición de columnas necesarias
final_cols = [
      'id', 'host_id', 'host_is_superhost', 'neighbourhood_cleansed', 
      'neighbourhood_group_cleansed', 'latitude', 'longitude', 
      'property_type', 'room_type', 'accommodates', 'bathrooms_text', 
      'bedrooms', 'price', 'minimum_nights', 'number_of_reviews', 
      'review_scores_rating', 'license', 'instant_bookable',
      'availability_365', 'calculated_host_listings_count', 
      'reviews_per_month', 'amenities'
  ]

ruta_listings = os.path.join(PATH_DATA, 'listings.csv')

if os.path.exists(ruta_listings):
      df_listings = pd.read_csv(ruta_listings, usecols=final_cols, low_memory=False)
      
      # Limpieza de precio: de "$1,200.00" a 1200.0
      df_listings['price'] = df_listings['price'].replace(r'[\$,]', '', regex=True).astype(float)
      
      print(f"✅ Listings procesados. Dimensiones: {df_listings.shape}")
else:
      print(f"❌ Error: No se encuentra listings.csv")

✅ Listings procesados. Dimensiones: (8215, 22)


ETL del archivo calendar.csv

In [5]:
cols_calendar = ['listing_id', 'date', 'available', 'minimum_nights', 'maximum_nights']
ruta_calendar = os.path.join(PATH_DATA, 'calendar.csv.zip')

if os.path.exists(ruta_calendar):
    df_calendar = pd.read_csv(ruta_calendar, usecols=cols_calendar, low_memory=False, compression='zip')
    
    df_calendar['date'] = pd.to_datetime(df_calendar['date'])
    df_calendar['available'] = df_calendar['available'].map({'t': True, 'f': False})
    df_calendar = df_calendar.sort_values(by=['listing_id', 'date'])
    
    # Guardamos por separado ya que es un volumen de datos distinto
    df_calendar.to_csv(os.path.join(PATH_DATA, 'calendar_limpio.csv.zip'), index=False, compression='zip')
    print(f"✅ Calendario procesado y guardado. Dimensiones: {df_calendar.shape}")

✅ Calendario procesado y guardado. Dimensiones: (2998475, 5)


ETL del archivo de rentas

In [6]:
ruta_renta_raw = os.path.join(PATH_DATA, 'renta_provincia_sevilla.csv')

if os.path.exists(ruta_renta_raw):
    df_renta_raw = pd.read_csv(ruta_renta_raw, sep=';', encoding='latin-1', decimal=',')
    
    # Filtrado: Sevilla Capital (Código 41091) y limpieza
    df_renta = df_renta_raw[
        (df_renta_raw['Secciones'].notna()) & 
        (df_renta_raw['Secciones'].astype(str).str.contains('^41091'))
    ].copy()
    
    if df_renta['Total'].dtype == 'object':
        df_renta['Total'] = df_renta['Total'].str.replace('.', '', regex=False).astype(float)
    
    df_renta.to_csv(os.path.join(PATH_DATA, 'renta_sevilla_capital_limpio.csv'), index=False)
    print(f"✅ Renta de Sevilla Capital lista. Secciones: {len(df_renta)}")

✅ Renta de Sevilla Capital lista. Secciones: 541


Exportación final

In [7]:
# 1. Convertir Airbnb a Geodataframe
df_renta['renta_media'] = df_renta['Total'].str.replace('.', '', regex=False).astype(float)

gdf_listings = gpd.GeoDataFrame(
    df_listings, 
    geometry=gpd.points_from_xy(df_listings.longitude, df_listings.latitude), 
    crs="EPSG:4326"
).to_crs(mapa_sevilla.crs)

# 2. Join Espacial  
pisos_con_seccion = gpd.sjoin(gdf_listings, mapa_sevilla[['CUSEC', 'geometry']], how="left", predicate="intersects")

# 3. Merge con Renta
df_renta['Secciones'] = df_renta['Secciones'].astype(str)
pisos_con_seccion['CUSEC'] = pisos_con_seccion['CUSEC'].astype(str)

dataset_final = pd.merge(pisos_con_seccion, df_renta[['Secciones', 'renta_media']], 
                        left_on='CUSEC', right_on='Secciones', how='left')

# 4. Guardar dataset
dataset_final.to_csv(os.path.join(PATH_DATA, 'dataset_final.csv'), index=False)
print("🏁 ¡HECHO! Tienes el dataset definitivo en 'dataset_final.csv'")

🏁 ¡HECHO! Tienes el dataset definitivo en 'dataset_final.csv'


In [8]:
# ==============================================================================
# CALCULAR VARIABLE OBJETIVO: TASA DE OCUPACIÓN
# ==============================================================================
print("📈 Calculando tasa de ocupación desde el calendario...")

# Agrupamos por piso y calculamos la media de 'available'
# Como 'available' es True (libre) y False (ocupado), la ocupación es: 1 - media_disponibilidad
df_ocupacion = df_calendar.groupby('listing_id')['available'].mean().reset_index()
df_ocupacion.columns = ['listing_id', 'tasa_disponibilidad']
df_ocupacion['tasa_ocupacion'] = 1 - df_ocupacion['tasa_disponibilidad']

# Nos quedamos solo con el ID y la Tasa de Ocupación
df_ocupacion = df_ocupacion[['listing_id', 'tasa_ocupacion']]
print("✅ Tasa de ocupación calculada.")

📈 Calculando tasa de ocupación desde el calendario...
✅ Tasa de ocupación calculada.


In [9]:
# ==============================================================================
# CARGAR RESULTADOS DE NLP (SENTIMIENTO)
# ==============================================================================
ruta_nlp = os.path.join(PATH_DATA, 'reviews_scores_nlp.csv')

if os.path.exists(ruta_nlp):
    df_nlp = pd.read_csv(ruta_nlp)
    print(f"✅ Scores de sentimiento cargados: {len(df_nlp)} registros.")
else:
    print("⚠️ Aviso: No se encuentra 'reviews_scores_nlp.csv'. Ejecuta primero el proceso de NLP.")

✅ Scores de sentimiento cargados: 7474 registros.


In [10]:
# ==============================================================================
# EXPORTACIÓN FINAL: DATASET_IA_FINAL.CSV
# ==============================================================================

# 1. Preparar Geodataframe (Lo que ya tenías)
gdf_listings = gpd.GeoDataFrame(
    df_listings, 
    geometry=gpd.points_from_xy(df_listings.longitude, df_listings.latitude), 
    crs="EPSG:4326"
).to_crs(mapa_sevilla.crs)

# 2. Unir con Secciones Censales (INE)
df_final = gpd.sjoin(gdf_listings, mapa_sevilla[['CUSEC', 'geometry']], how="left", predicate="intersects")

# 3. Unir con Renta
df_renta['Secciones'] = df_renta['Secciones'].astype(str)
df_final['CUSEC'] = df_final['CUSEC'].astype(str)
df_final = pd.merge(df_final, df_renta[['Secciones', 'renta_media']], left_on='CUSEC', right_on='Secciones', how='left')

# 4. Unir con Tasa de Ocupación (Calculada en Celda A)
df_final = pd.merge(df_final, df_ocupacion, left_on='id', right_on='listing_id', how='left')

# 5. Unir con NLP (Cargado en Celda B)
df_final = pd.merge(df_final, df_nlp, left_on='id', right_on='listing_id', how='left')

# 6. Limpieza de columnas duplicadas del merge
df_final = df_final.drop(columns=['listing_id_x', 'listing_id_y', 'Secciones'], errors='ignore')

# 7. GUARDAR EL DATASET PARA LA IA
ruta_ia = os.path.join(PATH_DATA, 'dataset_ia_final.csv')
df_final.to_csv(ruta_ia, index=False)

print(f"🏁 ¡TODO LISTO! El archivo '{ruta_ia}' contiene ahora:")
print("- Características del piso\n- Renta del barrio\n- Tasa de ocupación (Target)\n- Sentimiento de las reseñas")

🏁 ¡TODO LISTO! El archivo '..\data\dataset_ia_final.csv' contiene ahora:
- Características del piso
- Renta del barrio
- Tasa de ocupación (Target)
- Sentimiento de las reseñas


In [11]:
print(f"Total inicial en listings: {len(df_listings)}")

# Tras el cruce con el mapa
print(f"Tras cruce con mapa: {len(df_final)}") 

# Tras el cruce con ocupación
print(f"Tras cruce con ocupación: {len(df_final.dropna(subset=['tasa_ocupacion']))}")

# Tras el cruce con NLP
print(f"Tras cruce con sentimiento: {len(df_final.dropna(subset=['score_sentimiento']))}")

Total inicial en listings: 8215
Tras cruce con mapa: 8215
Tras cruce con ocupación: 8215
Tras cruce con sentimiento: 7474
